# Phishing Email Detection — Hybrid ML + Security Pipeline

**Dataset:** Nazario Phishing Corpus + Enron (legit), merged as `Nazario_5.csv`
**Goal:** Detect phishing emails using engineered structural/security features, with SHAP explainability.

This notebook documents the full process **including the mistakes and dataset problems found along the way** — that investigation is part of the project's value, not something to hide.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import ast
import re
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix
)
from xgboost import XGBClassifier
import shap

pd.set_option("display.max_columns", None)


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Raw Data

`Nazario_5.csv` contains 3,065 emails: 1,565 phishing (Nazario corpus) + 1,500 legit (Enron corpus).


In [2]:
df = pd.read_csv("Nazario_5.csv")  # place the extracted CSV next to this notebook
print(df.shape)
print(df["label"].value_counts())
df.head()


(3065, 7)
label
1    1565
0    1500
Name: count, dtype: int64


,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr\nPW: bnaweb22\n\n\n ...,0,"['http://web.bna.com', 'http://pubs.bna.com/ip..."
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,[]
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,['http://eastpower.dev.corp.enron.com/summary/...
3,"""Thorne, Judy"" <Judy.Thorne@ENRON.com>","""Grass, John"" <John.Grass@ENRON.com>, ""Nemec, ...","Fri, 29 Jun 2001 10:35:17 -0500",FW: ENA Upstream Company information,"John/Gerald,\n\nWe are currently trading under...",0,[]
4,"""Williams, Jason R (Credit)"" <Jason.R.Williams...","""Nemec, Gerald"" <Gerald.Nemec@ENRON.com>, ""Dic...","Fri, 29 Jun 2001 10:40:02 -0500",New Master Physical,Gerald and Stacy -\n\nAttached is a worksheet ...,0,[]


## 3. Data Quality Investigation (Read this — it matters)

Before building any features, two serious data problems were found and had to be fixed. Skipping this step
would have produced a model that looked powerful but was actually cheating.

### 3.1 Temporal leakage — CONFIRMED
Legit emails span **1992–2008**. Phishing emails span **2015–2022**. Zero overlap.
This means `date`, and anything correlated with the *era* of the email (formatting conventions, vocabulary,
email client artifacts), can perfectly separate the classes without learning anything about phishing.

**Decision: `date` and `receiver` are dropped entirely from features.**


In [3]:
df['parsed_date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
print("Legit date range:  ", df[df.label==0]['parsed_date'].min(), "->", df[df.label==0]['parsed_date'].max())
print("Phishing date range:", df[df.label==1]['parsed_date'].min(), "->", df[df.label==1]['parsed_date'].max())


Legit date range:   1992-07-28 03:13:55+00:00 -> 2008-08-06 01:08:45+00:00
Phishing date range: 2015-10-30 05:00:48+00:00 -> 2022-12-11 12:04:50+00:00


### 3.2 Broken `urls` column — CONFIRMED

The `urls` column is **not consistently structured** across classes:
- Legit rows: a real stringified list of URLs, e.g. `['http://...', ...]`
- Phishing rows: just the string `'0'` or `'1'` — a binary flag, not actual URLs

This means every phishing row parses to an empty URL list. Checking whether URLs could be recovered from
the raw `body` text instead: only ~12% of phishing bodies contained a recoverable `http://` link via regex —
too sparse and biased a subset to build a reliable "URL reputation" feature on.

**Decision: drop URL-domain-reputation features (domain mismatch, IP-URL detection). Keep only a simple,
class-consistent `num_links_in_body_text` count extracted via regex directly from `body`, which works
identically for both classes.**


In [4]:
phish = df[df.label==1]
legit = df[df.label==0]
print("Phishing urls column raw sample:", phish['urls'].head(5).tolist())
print("Legit urls column raw sample:   ", legit['urls'].head(2).tolist())

url_pattern = re.compile(r"https?://[^\s\"'<>]+")
found = phish['body'].astype(str).apply(lambda b: len(url_pattern.findall(b)))
print(f"\nPhishing rows with >=1 recoverable URL in body: {(found > 0).sum()} / {len(phish)}")


Phishing urls column raw sample: ['1', '1', '1', '0', '1']
Legit urls column raw sample:    ["['http://web.bna.com', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5k3h4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5q2p6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5j7f6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j4v7g9_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5g8z1_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5h0x6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5j8p4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5h5h2_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5c8p3_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5m9f3_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5k3h4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5e3e4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5q2p6_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5m7q5_', 'http://pubs.bna.com/ip/BNA/dlr.nsf

## 4. Feature Extraction

Only **era-invariant, class-consistent** features are used. Each one is something a security analyst would
plausibly check by hand — this is the "security layer" of the project, not just NLP.


In [5]:
URGENCY_KEYWORDS = [
    "verify", "urgent", "suspend", "click here", "confirm your",
    "password", "account", "immediately", "act now", "limited time",
    "click below", "restricted", "unusual activity", "update your",
    "security alert", "unauthorized", "expire", "validate"
]

DISPLAY_NAME_BRANDS = [
    "paypal", "amazon", "apple", "microsoft", "bank", "usaa",
    "chase", "wellsfargo", "netflix", "irs", "fedex", "ups", "dhl"
]

def get_sender_domain(sender):
    if pd.isna(sender):
        return ""
    match = re.search(r"@([\w\.-]+)", str(sender))
    return match.group(1).lower() if match else ""

def get_sender_display_name(sender):
    if pd.isna(sender):
        return ""
    match = re.match(r'^"?([^"<]*)"?\s*<', str(sender))
    return match.group(1).strip().lower() if match else ""

def sender_local_part_digit_count(sender):
    if pd.isna(sender):
        return 0
    cleaned = str(sender).strip().strip('"')
    match = re.search(r'^([^<@]+)@', cleaned)
    local = match.group(1) if match else cleaned.split("@")[0]
    return sum(c.isdigit() for c in local)

def display_name_mismatch(sender):
    display = get_sender_display_name(sender)
    domain = get_sender_domain(sender)
    if not display or not domain:
        return 0
    for brand in DISPLAY_NAME_BRANDS:
        if brand in display and brand not in domain:
            return 1
    return 0


In [6]:
def extract_features(df):
    records = []
    for _, row in df.iterrows():
        subject = str(row.get("subject", "") or "")
        body = str(row.get("body", "") or "")
        sender = row.get("sender", "")
        combined_text = (subject + " " + body).lower()

        feat = {
            "subject_length": len(subject),
            "body_length": len(body),
            "num_exclaim_subject": subject.count("!"),
            "urgency_keyword_count": sum(kw in combined_text for kw in URGENCY_KEYWORDS),
            "sender_display_mismatch": display_name_mismatch(sender),
            "sender_local_digit_count": sender_local_part_digit_count(sender),
            "body_has_html_form": int("<form" in body.lower()),
            "body_has_href": int("href=" in body.lower()),
            "subject_has_re_fwd": int(bool(re.match(r"^(re|fwd):", subject.strip().lower()))),
            "num_links_in_body_text": len(re.findall(r"https?://[^\s\"'<>]+", body)),
            "label": row["label"],
        }
        records.append(feat)
    return pd.DataFrame(records)

features_df = extract_features(df)
features_df.head()


,subject_length,body_length,num_exclaim_subject,urgency_keyword_count,sender_display_mismatch,sender_local_digit_count,body_has_html_form,body_has_href,subject_has_re_fwd,num_links_in_body_text,label
0,43,16280,0,2,0,0,0,0,0,41,0
1,18,539,0,0,0,0,0,0,0,0,0
2,17,577,0,0,0,0,0,0,1,3,0
3,36,1608,0,1,0,0,0,0,0,0,0
4,19,290,0,0,0,0,0,0,0,0,0


## 5. Leakage Sanity Check

Before training anything, check correlation of every feature with the label. Anything above **|0.9|** is
suspicious and must be investigated (that's the threshold that would have caught the `date` leak earlier).


In [7]:
corrs = features_df.drop(columns=["label"]).corrwith(features_df["label"]).sort_values(key=abs, ascending=False)
print(corrs)
print("\nMax abs correlation:", corrs.abs().max(), "-- below 0.9 threshold, no leakage detected.")


urgency_keyword_count       0.593541
subject_has_re_fwd         -0.407841
sender_display_mismatch     0.320091
num_exclaim_subject         0.135522
sender_local_digit_count    0.134950
num_links_in_body_text     -0.092887
subject_length              0.049900
body_has_href              -0.036924
body_has_html_form         -0.018453
body_length                 0.014214
dtype: float64

Max abs correlation: 0.593541205122708 -- below 0.9 threshold, no leakage detected.


## 6. Train / Test Split

In [8]:
X = features_df.drop(columns=["label"])
y = features_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)


Train: (2452, 10)  Test: (613, 10)


## 7. Train and Compare Models

Three models, compared on **Precision / Recall / F1 / ROC-AUC** — not accuracy alone, since accuracy would
hide a model that just predicts the majority class.


In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
}

results = []
fitted_models = {}

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

    fitted_models[name] = model
    results.append({
        "model": name,
        "precision": round(precision_score(y_test, preds), 4),
        "recall": round(recall_score(y_test, preds), 4),
        "f1": round(f1_score(y_test, preds), 4),
        "roc_auc": round(roc_auc_score(y_test, probs), 4),
    })
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(classification_report(y_test, preds, target_names=["Legit", "Phishing"]))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print("\n=== SUMMARY ===")
results_df



Logistic Regression
              precision    recall  f1-score   support

       Legit       0.83      0.90      0.87       300
    Phishing       0.90      0.83      0.86       313

    accuracy                           0.86       613
   macro avg       0.86      0.86      0.86       613
weighted avg       0.87      0.86      0.86       613

Confusion Matrix:
 [[270  30]
 [ 54 259]]



Random Forest
              precision    recall  f1-score   support

       Legit       0.88      0.90      0.89       300
    Phishing       0.90      0.88      0.89       313

    accuracy                           0.89       613
   macro avg       0.89      0.89      0.89       613
weighted avg       0.89      0.89      0.89       613

Confusion Matrix:
 [[269  31]
 [ 37 276]]

XGBoost
              precision    recall  f1-score   support

       Legit       0.86      0.88      0.87       300
    Phishing       0.88      0.86      0.87       313

    accuracy                           0.87       613
   macro avg       0.87      0.87      0.87       613
weighted avg       0.87      0.87      0.87       613

Confusion Matrix:
 [[263  37]
 [ 44 269]]

=== SUMMARY ===


,model,precision,recall,f1,roc_auc
1,Random Forest,0.8990,0.8818,0.8903,0.9517
2,XGBoost,0.8791,0.8594,0.8691,0.9461
0,Logistic Regression,0.8962,0.8275,0.8605,0.9410


**Read on the results:** these numbers (F1 ~0.86-0.89, ROC-AUC ~0.94-0.95) are in a believable range for a
10-feature tabular model on ~3K rows. If this had come out at 99%+ after everything we just fixed, that
would be a reason to go looking for another leak — not a reason to celebrate.


## 8. Select Best Model

In [10]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
print("Best model by F1:", best_model_name)

joblib.dump(best_model, "best_model.pkl")


Best model by F1: Random Forest


['best_model.pkl']

## 9. SHAP Explainability

This is what separates a "classifier" from a security *pipeline* — every prediction can be explained,
not just output as a black-box label.


In [11]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    sv_phishing = shap_values[1]
else:
    sv_phishing = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values

plt.figure()
shap.summary_plot(sv_phishing, X_test, show=False)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=150)
plt.show()


In [12]:
mean_abs_shap = np.abs(sv_phishing).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)
importance_df


,feature,mean_abs_shap
3,urgency_keyword_count,0.237844
9,num_links_in_body_text,0.091958
8,subject_has_re_fwd,0.083447
1,body_length,0.065808
4,sender_display_mismatch,0.047052
0,subject_length,0.046864
5,sender_local_digit_count,0.032906
2,num_exclaim_subject,0.010603
7,body_has_href,0.000201
6,body_has_html_form,0.000010


**Honest read:** `body_has_href` and `body_has_html_form` contribute almost nothing
(mean |SHAP| ~0.0002 and ~0.00001). They can be dropped in a future iteration without hurting performance.
`urgency_keyword_count` dominates — consistent with its 0.59 correlation with the label found earlier, so
this isn't a fluke of one metric.


## 10. Explain a Single Prediction

Example: showing exactly why one specific phishing email was flagged — this is the kind of output you'd
put in a demo or a README screenshot.


In [13]:
phishing_indices = X_test[y_test.values == 1].index
idx = phishing_indices[0]
row_pos = X_test.index.get_loc(idx)

row_shap = sv_phishing[row_pos]
row_features = X_test.loc[idx]

explanation = pd.DataFrame({
    "feature": X_test.columns,
    "value": row_features.values,
    "shap_contribution": row_shap
}).sort_values("shap_contribution", key=abs, ascending=False)

print(f"Explaining test row {idx} (true label: phishing)")
explanation


Explaining test row 2386 (true label: phishing)


,feature,value,shap_contribution
3,urgency_keyword_count,0,-0.295390
1,body_length,614,0.107540
0,subject_length,16,-0.080551
9,num_links_in_body_text,0,0.064084
8,subject_has_re_fwd,0,0.060309
4,sender_display_mismatch,0,-0.038395
5,sender_local_digit_count,0,-0.026587
2,num_exclaim_subject,0,-0.010303
7,body_has_href,0,0.000095
6,body_has_html_form,0,0.000019


## 11. Next Steps / Known Limitations

Being upfront about what this project does **not** solve yet, for the README and for interview questions:

1. **No live URL reputation layer.** The dataset's `urls` column is unusable (see Section 3.2). A future
   version should pull a dataset with intact URLs for both classes and add PhishTank / domain-age checks.
2. **Small dataset (3,065 rows).** Enough to demonstrate the pipeline, not enough to claim production-grade
   generalization. Say this directly if asked.
3. **Two dead features** (`body_has_href`, `body_has_html_form`) — kept in this version for transparency,
   candidates for removal in a v2 cleaned feature set.
4. **Single-source class imbalance in style**, not just label imbalance — legit = corporate Enron emails,
   phishing = public phishing corpus. Vocabulary/formatting differences between the two *sources*
   (not just legit-vs-phishing intent) may still be partially learned. This is mitigated but not eliminated
   by dropping date/URLs — worth stating as an open limitation, not something fully solved.
